# Clonotrace Python Demo

This notebook demonstrates the Clonotrace Python package on the hematopoiesis dataset from [Weinreb et al. 2020](https://www.science.org/doi/10.1126/science.aaw3381).

**Dataset**: 34,782 cells sampled at 3 time points, with 802 expanded clones (10+ cells).

Each step is timed for benchmarking against the R implementation.

In [1]:
import time
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmread

# Timing results dict
timings = {}

def timed(name):
    """Context manager to time a step."""
    class Timer:
        def __enter__(self):
            self.t0 = time.perf_counter()
            return self
        def __exit__(self, *args):
            elapsed = time.perf_counter() - self.t0
            timings[name] = {"time": elapsed}
            print(f"  {name}: {elapsed:.2f}s")
    return Timer()

## Step 0: Load Data

Load the extracted data (PCA embeddings, cell metadata, expression matrix).
The data was extracted from the Seurat RDS file using the companion R script.

In [2]:
DATA_DIR = "real_data"

# PCA embeddings
pca_df = pd.read_csv(f"{DATA_DIR}/pca.csv", index_col=0)
pca = pca_df.values
cell_names = pca_df.index.tolist()

# UMAP coordinates
umap_df = pd.read_csv(f"{DATA_DIR}/umap.csv", index_col=0)

# Cell metadata
cell_meta = pd.read_csv(f"{DATA_DIR}/cell_meta.csv", index_col=0)

# Expression matrix (genes x cells, sparse)
exprs = mmread(f"{DATA_DIR}/exprs.mtx").tocsr()
gene_names = open(f"{DATA_DIR}/gene_names.txt").read().strip().split("\n")
cell_names_exprs = open(f"{DATA_DIR}/cell_names_exprs.txt").read().strip().split("\n")

print(f"Cells: {pca.shape[0]}, PCA dims: {pca.shape[1]}, Genes: {len(gene_names)}")
print(f"Expression matrix: {exprs.shape}")
print(f"Cell types: {cell_meta['Cell.type.annotation'].nunique()}")
print(f"Clusters: {cell_meta['cluster'].nunique()}")

Cells: 34782, PCA dims: 50, Genes: 25289
Expression matrix: (25289, 34782)
Cell types: 3
Clusters: 9


## Step 1: Build Cell kNN Graph

Construct a k-nearest neighbor graph from PCA embeddings and row-normalize to get a transition matrix.

In [3]:
from clonotrace.auxiliary import embedding2knn

with timed("embedding2knn"):
    cell_knn = embedding2knn(pca, k=30, mode="connectivity")

# Row-normalize to transition matrix (compute_transition equivalent)
with timed("compute_transition"):
    row_sums = np.array(cell_knn.sum(axis=1)).ravel()
    row_sums = np.maximum(row_sums, 1e-12)
    D_inv = sp.diags(1.0 / row_sums)
    cell_knn = D_inv.dot(cell_knn)

print(f"kNN shape: {cell_knn.shape}, nnz: {cell_knn.nnz}")

  embedding2knn: 0.51s
  compute_transition: 0.01s
kNN shape: (34782, 34782), nnz: 1793740


## Step 2: Prepare Clone Labels

Filter to expanded clones (10+ cells) and set non-expanded clones to NaN.

In [4]:
clone_counts = cell_meta["clone"].value_counts()
expanded_clones = clone_counts[clone_counts >= 10].index.tolist()

clone_labels = cell_meta["clone"].copy()
clone_labels[~clone_labels.isin(expanded_clones)] = np.nan

print(f"Total clones: {cell_meta['clone'].nunique()}")
print(f"Expanded clones (>=10 cells): {len(expanded_clones)}")
print(f"Labeled cells: {clone_labels.notna().sum()} / {len(clone_labels)}")

Total clones: 8108
Expanded clones (>=10 cells): 802
Labeled cells: 15801 / 34782


## Step 3: Label Spreading Bootstrap

Propagate clone labels across the cell graph with bootstrap stability estimation.

In [5]:
from clonotrace.label_propagation import label_spreading_bootstrap

with timed("label_spreading_bootstrap"):
    result = label_spreading_bootstrap(
        adj=cell_knn,
        labels=clone_labels.values,
        alpha=0.6,
        sample_rate=0.8,
        sample_n=48,
        n_jobs=1,  # single-core for fair comparison
    )

cell_clone_prob_raw = result["prob"]
deviance = result["deviance"]

print(f"Clone prob shape: {cell_clone_prob_raw.shape}")
print(f"Deviance range: [{deviance.min():.3f}, {deviance.max():.3f}]")

  label_spreading_bootstrap: 203.81s
Clone prob shape: (34782, 802)
Deviance range: [0.005, 0.407]


## Step 4: Filter by Deviance + Sparsify

Remove cells with high label propagation deviance (>0.3) and sparsify the clone probability matrix.

In [6]:
from clonotrace.auxiliary import mat_sparsify

with timed("filter_sparsify"):
    keep_mask = deviance < 0.3
    cell_clone_prob = cell_clone_prob_raw[keep_mask]
    
    # Row-normalize
    row_sums = cell_clone_prob.sum(axis=1, keepdims=True)
    row_sums = np.maximum(row_sums, 1e-12)
    cell_clone_prob = cell_clone_prob / row_sums
    
    # Sparsify
    cell_clone_prob = mat_sparsify(cell_clone_prob, row_mass=0.9, col_mass=0.9)
    
    # Re-normalize
    row_sums = cell_clone_prob.sum(axis=1, keepdims=True)
    row_sums = np.maximum(row_sums, 1e-12)
    cell_clone_prob = cell_clone_prob / row_sums
    
    # Convert to sparse
    cell_clone_prob = sp.csr_matrix(cell_clone_prob)

# Track which cells survived filtering
kept_cell_names = [cell_names[i] for i in range(len(cell_names)) if keep_mask[i]]

# Get clone names (same order as columns)
from pandas import factorize
valid_mask = pd.notna(clone_labels)
_, clone_name_list = factorize(clone_labels[valid_mask], sort=True)
clone_name_list = clone_name_list.tolist()

print(f"Cells after filtering: {cell_clone_prob.shape[0]}")
print(f"Clones: {cell_clone_prob.shape[1]}")
print(f"Sparsity: {cell_clone_prob.nnz / (cell_clone_prob.shape[0] * cell_clone_prob.shape[1]):.4f}")

  filter_sparsify: 2.40s
Cells after filtering: 32251
Clones: 802
Sparsity: 0.0502


## Step 5: Load Pre-computed Clone Distances

Clone-to-clone OT distances are expensive to compute. We load the pre-computed results.

In [7]:
from clonotrace.auxiliary import long2square

with timed("load_clone_dis"):
    clone_dis_long = pd.read_csv(f"{DATA_DIR}/clone_graph_dis.tsv", sep="\t")
    clone_dis = long2square(
        clone_dis_long,
        row_names_from="group1",
        col_names_from="group2",
        values_from="dis",
        symmetric=True
    )
    np.fill_diagonal(clone_dis, 0)
    # Fill NaN with max distance (for pairs not computed)
    max_dis = np.nanmax(clone_dis)
    clone_dis = np.nan_to_num(clone_dis, nan=max_dis)

print(f"Clone distance matrix: {clone_dis.shape}")
print(f"Distance range: [{clone_dis.min():.3f}, {clone_dis.max():.3f}]")

  load_clone_dis: 0.36s
Clone distance matrix: (802, 802)
Distance range: [0.000, 0.000]


## Step 6: Clone Clustering (Louvain)

Cluster clones using shared nearest neighbor graph + Louvain community detection.

In [8]:
from clonotrace.cluster import leiden_dis
import random

np.random.seed(1230)
random.seed(1230)

with timed("clone_clustering"):
    clone_cluster = leiden_dis(
        clone_dis, k=20, resolution=0.5,
        if_umap=True, method="louvain"
    )

print(f"Number of clone clusters: {clone_cluster['cluster'].nunique()}")
print(f"Cluster sizes:\\n{clone_cluster['cluster'].value_counts().sort_index()}")

/Users/yizhouw/miniconda3/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")


  clone_clustering: 3.83s
Number of clone clusters: 1
Cluster sizes:\ncluster
0    802
Name: count, dtype: int64


## Step 7: Clone-level Pseudotime

Compute diffusion pseudotime on the clone MDS embedding, starting from cluster "0".

In [9]:
from clonotrace.pseudotime import clone_dpt

# Load pre-computed MDS embedding
clone_embedding = pd.read_csv(f"{DATA_DIR}/clone_mds.tsv", sep="\t", index_col=0)

# Add clone names to clone_cluster for clone_dpt
clone_cluster_named = clone_cluster.copy()
clone_cluster_named.index = clone_name_list

with timed("clone_dpt"):
    clone_t = clone_dpt(
        clone_embedding=clone_embedding,
        cell_meta=cell_meta,
        clone_col="clone",
        cluster_col="cluster",
        start_cluster="0"
    )

clone_cluster_named["dpt"] = clone_t
print(f"Pseudotime range: [{clone_t.min():.3f}, {clone_t.max():.3f}]")

# Smooth pseudotime to cell level
cell_meta["cell_t"] = np.nan
cell_clone_prob_dense = cell_clone_prob.toarray()
cell_t_values = cell_clone_prob_dense @ clone_t
cell_meta.loc[kept_cell_names, "cell_t"] = cell_t_values

  clone_dpt: 0.15s
Pseudotime range: [0.000, 1.000]


/Users/yizhouw/Desktop/packages/Clonotrace_python/clonotrace/pseudotime.py:231: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


## Step 8: Profile Assignment

Map clone clusters to cell-level profiles using the clone probability matrix.

In [10]:
from clonotrace.auxiliary import long2sparse

with timed("profile_assignment"):
    # Build clone -> cluster indicator matrix
    n_clusters = clone_cluster["cluster"].nunique()
    cluster_labels = clone_cluster["cluster"].values
    clone_profile_mat = np.zeros((len(clone_name_list), n_clusters))
    for i, cl in enumerate(cluster_labels):
        clone_profile_mat[i, cl] = 1.0

    # Cell profile prob = cell_clone_prob @ clone_profile_mat
    cell_profile_prob = cell_clone_prob.dot(sp.csr_matrix(clone_profile_mat))
    cell_profile_prob = cell_profile_prob.toarray()

# Assign hard profile labels (max prob > 0.5)
cell_meta["profile"] = np.nan
profile_max = cell_profile_prob.max(axis=1)
profile_argmax = cell_profile_prob.argmax(axis=1)
hard_labels = np.where(profile_max > 0.5, profile_argmax, np.nan)
cell_meta.loc[kept_cell_names, "profile"] = hard_labels

print(f"Cell profile prob shape: {cell_profile_prob.shape}")
print(f"Cells with profile assignment: {np.sum(profile_max > 0.5)}")

  profile_assignment: 0.00s
Cell profile prob shape: (32251, 1)
Cells with profile assignment: 32251


## Step 9: Profile-Cluster Enrichment

Permutation test to identify which profiles are enriched in which cell clusters.

In [11]:
from clonotrace.profile_deg import cluster_profile_enrich

cluster_labels_for_enrich = cell_meta.loc[kept_cell_names, "cluster"].values

with timed("cluster_profile_enrich"):
    enrich = cluster_profile_enrich(
        cell_profile_prob,
        cluster_labels_for_enrich,
        permute_n=300
    )

print(f"Enrichment prob shape: {enrich['prob'].shape}")
print(f"Significant (p<0.05): {np.sum(enrich['pval'] < 0.05)}")

  cluster_profile_enrich: 0.22s
Enrichment prob shape: (9, 1)
Significant (p<0.05): 0


## Step 10: Profile-specific DEG

Detect differentially expressed genes for profile 1 in cluster 4.

In [12]:
from clonotrace.profile_deg import profile_cluster_DEG

# Build expression DataFrame with gene names as index, cell names as columns
exprs_df = pd.DataFrame.sparse.from_spmatrix(exprs, index=gene_names, columns=cell_names_exprs)

# Build cell_profile_prob as DataFrame
cell_profile_prob_df = pd.DataFrame(
    cell_profile_prob,
    index=kept_cell_names,
    columns=[str(i) for i in range(cell_profile_prob.shape[1])]
)

with timed("profile_cluster_DEG"):
    DEG_result = profile_cluster_DEG(
        profile="0",  # 0-indexed in Python (= profile "1" in R)
        cluster=4,    # Match R cluster 4
        exprs=exprs_df,
        cell_meta=cell_meta,
        cell_profile_prob=cell_profile_prob_df,
        cluster_col="cluster",
        pseudotime_col="cell_t",
        permute_n=50
    )

if DEG_result is not None:
    sig_genes = DEG_result["stat"][DEG_result["stat"]["padj"] < 0.05]
    print(f"Significant DEGs (padj < 0.05): {len(sig_genes)}")
else:
    print("No DEG result (too few cells in this cluster/profile combination)")

  profile_cluster_DEG: 0.01s
No DEG result (too few cells in this cluster/profile combination)


/Users/yizhouw/Desktop/packages/Clonotrace_python/clonotrace/profile_deg.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = (df.groupby("bin")[["target", "control"]]


## Timing Summary

Save all timing results for comparison with R.

In [13]:
print("=" * 50)
print("Python Pipeline Timing Summary")
print("=" * 50)
total = 0
for name, data in timings.items():
    t = data["time"]
    total += t
    print(f"  {name:<30s} {t:>8.2f}s")
print(f"  {'TOTAL':<30s} {total:>8.2f}s")

# Save results
with open("real_data_python_results.json", "w") as f:
    json.dump(timings, f, indent=2)
print(f"\nResults saved to real_data_python_results.json")

Python Pipeline Timing Summary
  embedding2knn                      0.51s
  compute_transition                 0.01s
  label_spreading_bootstrap        203.81s
  filter_sparsify                    2.40s
  load_clone_dis                     0.36s
  clone_clustering                   3.83s
  clone_dpt                          0.15s
  profile_assignment                 0.00s
  cluster_profile_enrich             0.22s
  profile_cluster_DEG                0.01s
  TOTAL                            211.29s

Results saved to real_data_python_results.json
